# CyDSP — affinage L4 avec transfert direct des fichiers

Ce notebook fonctionne **sans connexion ni montage Google Drive**.

1. Importer ce fichier **CyDSP_HRTF_L4_Direct.ipynb** dans Colab.
2. Choisir **GPU → L4**, puis **Exécution → Tout exécuter**.
3. Quand le sélecteur de fichiers apparaît, choisir sur ton PC le fichier
   **CyDSP-HRTF-L4-Direct.zip**, sans le décompresser.
4. À la fin, récupérer le fichier **CyDSP-HRTF-L4-Results.zip** téléchargé par
   le notebook. Il contient les meilleurs poids, la reprise et les rapports.

Le modèle repart du meilleur **D2/KEMAR 48 kHz / 24 filtres** compatible avec
CyDSP. Même architecture, batch 512, 140 000 étapes supplémentaires et taux
Adam/cosine réduit de **0,0003 à 0,00001**. Environ **30 minutes de calcul estimées**
sur la L4, hors installation, téléchargements, validation et export.
Le meilleur modèle initial reste candidat si la validation ne s'améliore pas.

Les checkpoints sont enregistrés **dans la session Colab**. Télécharger le ZIP
avant de fermer/supprimer cette session. La dernière cellule peut aussi être
exécutée manuellement pour récupérer un checkpoint partiel après interruption.
Une perte de la machine Colab efface les fichiers qui n'ont pas été téléchargés.

Après une coupure, la même machine peut reprendre ses checkpoints locaux.
Sur une nouvelle machine, mettre RESTORE_RESULT=True puis fournir un ZIP de
résultats déjà téléchargé quand la cellule de reprise le demande.

Code original **MIT**, données SADIE II **Apache-2.0**, notices incluses.
Modèle d'un auditeur connu : amplitude et ITD basse fréquence, sans phase HRIR
complète. Le test directionnel D2 a déjà été évalué ; aucune nouvelle indépendance
statistique ni amélioration de qualité ne sont présumées.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import subprocess
import sys

EXPERIMENT = "l4-hrtf-48k-direct-001"       # Nouveau nom pour un plan différent.
SAMPLE_RATE = 48_000
LEARNING_RATE = 0.0003
MINIMUM_LEARNING_RATE = 0.00001
REQUIRE_L4 = True
SUBJECTS = ["D2"]                 # Ce profil affine D2/KEMAR à 48 kHz.
FILTER_BUDGETS = [24]             # Référence fournie : 24 filtres à 48 kHz.
STEPS = 140_000                   # Calendrier complet, inchangé à la reprise.
BATCH_SIZE = 512                  # Mesurer avant de modifier ce réglage.
VALIDATION_EVERY = 1_000
WARM_START_D2 = True              # False : repartir de zéro.
RUN_BATCH_BENCHMARK = False
RUN_FINAL_EVALUATION = True
DOWNLOAD_ALL_SUBJECTS = False     # Télécharger 20 sujets ne crée pas un modèle multi-auditeur.
CACHE_ARCHIVES_LOCAL = False
SAVE_EVERY_SECONDS = 60
REQUIRE_A100 = False              # L4 acceptée ; True exige spécifiquement une A100.
RESULTS_ROOT = Path("/content/CyDSP-HRTF-results")
BUNDLE_PATH = Path("/content/CyDSP-HRTF-L4-Direct.zip")
EXPECTED_BUNDLE_SHA256 = "0cad21d001f2c946938ad3a77154008c3f73363a629f7d5366919128d5b9a639"

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", EXPERIMENT):
    raise ValueError("Nom d'expérience invalide")
if not SUBJECTS or len(set(SUBJECTS)) != len(SUBJECTS):
    raise ValueError("Choisir des sujets distincts")
if not FILTER_BUDGETS or len(set(FILTER_BUDGETS)) != len(FILTER_BUDGETS) or set(FILTER_BUDGETS) - {24}:
    raise ValueError("Ce profil 48 kHz inclut le modèle à 24 filtres")
if not 100 <= STEPS <= 1_000_000 or not 1 <= BATCH_SIZE <= 4096:
    raise ValueError("STEPS : 100..1000000 ; BATCH_SIZE : 1..4096")
if VALIDATION_EVERY < 1 or SAVE_EVERY_SECONDS < 10:
    raise ValueError("Intervalles de sauvegarde invalides")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Environnement prévu pour Python 3.11 à 3.13")
print("Plan :", SUBJECTS, FILTER_BUDGETS, STEPS, "étapes par modèle")

if SAMPLE_RATE != 48000 or SUBJECTS != ["D2"]:
    raise ValueError("Ce profil part du modèle D2 48 kHz ; créer une autre expérience pour changer de cible")
if not 0 < MINIMUM_LEARNING_RATE <= LEARNING_RATE <= 0.1:
    raise ValueError("Vérifier les taux d’apprentissage")
print("Affinage", SAMPLE_RATE, "Hz ; LR", LEARNING_RATE, "→", MINIMUM_LEARNING_RATE)

RESTORE_RESULT = False  # True : fournir un ZIP de résultats téléchargé avant une coupure.


## 1. Envoyer le fichier ZIP directement dans Colab

Choisir le fichier ZIP sur ton PC. Cette étape ne demande aucune autorisation Drive.


In [ ]:
from google.colab import files

if not BUNDLE_PATH.is_file():
    print("Choisir sur ton PC : CyDSP-HRTF-L4-Direct.zip (sans le décompresser).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choisir uniquement le fichier ZIP fourni avec ce notebook.")
    blob = next(iter(uploaded.values()))
    if len(blob) > 64 * 1024 * 1024 or hashlib.sha256(blob).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise ValueError("Mauvais ZIP : utiliser CyDSP-HRTF-L4-Direct.zip fourni avec ce notebook.")
    BUNDLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    BUNDLE_PATH.write_bytes(blob)
print("Paquet présent dans la session Colab :", BUNDLE_PATH)


## 2. Vérifier et préparer les sources
Le SHA-256 associe ce notebook à son paquet. Chemins, tailles et empreintes
sont vérifiés avant extraction. Aucun clonage privé ou transfert du projet
complet n'est nécessaire.


In [ ]:
import io
import stat
import zipfile
from pathlib import PurePosixPath

blob = BUNDLE_PATH.read_bytes()
if len(blob) > 64 * 1024 * 1024 or hashlib.sha256(blob).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise ValueError("Le paquet ne correspond pas à ce notebook (SHA-256)")
ROOT = Path("/content") / ("CyDSP-HRTF-" + EXPECTED_BUNDLE_SHA256[:12])
payload = {}
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    entries = archive.infolist()
    names = [e.filename for e in entries]
    if len(names) > 121 or len(set(names)) != len(names) or "manifest.json" not in names:
        raise ValueError("Membres du paquet invalides")
    if sum(e.file_size for e in entries) > 64 * 1024 * 1024:
        raise ValueError("Paquet décompressé trop grand")
    for entry in entries:
        name = entry.filename
        path = PurePosixPath(name)
        if (entry.is_dir() or entry.file_size > 16 * 1024 * 1024
                or stat.S_ISLNK(entry.external_attr >> 16)
                or "\\" in name or ":" in name or path.is_absolute()
                or any(p in ("", ".", "..") for p in name.split("/"))
                or path.as_posix() != name):
            raise ValueError("Chemin ou taille de membre invalide")
    manifest = json.loads(archive.read("manifest.json"))
    if manifest.get("schema") != 1 or manifest.get("kind") != "cydsp-hrtf-colab-source":
        raise ValueError("Type de paquet incorrect")
    if set(names) != set(manifest["files"]) | {"manifest.json"}:
        raise ValueError("Contenu différent du manifeste")
    for name, record in manifest["files"].items():
        value = archive.read(name)
        if len(value) != record["bytes"] or hashlib.sha256(value).hexdigest() != record["sha256"]:
            raise ValueError("Empreinte incorrecte : " + name)
        target = ROOT.joinpath(*PurePosixPath(name).parts)
        if not target.resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Chemin en dehors du projet")
        if target.exists() and target.read_bytes() != value:
            raise ValueError("Source locale modifiée : " + name)
        payload[name] = value
for name, value in payload.items():
    target = ROOT.joinpath(*PurePosixPath(name).parts)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(ROOT / "scripts"))
from colab_hrtf import (
    atomic_bytes, atomic_json, digest, read_json, restore_run,
    run_with_backup, snapshot_run,
)
REFERENCES = read_json(ROOT / "colab-references.json")
catalog = read_json(ROOT / "data/hrtf/training-sources.json")
if set(SUBJECTS) - {s["subject"] for s in catalog["subjects"]}:
    raise ValueError("Sujet absent du catalogue SADIE II")
print("Sources prêtes :", ROOT)

from colab_hrtf import (
    run_with_backup as _run_with_backup, snapshot_run as _snapshot_run,
    export_direct_result, restore_direct_result,
)
def run_with_backup(command, root, run, destination, seconds=60):
    return _run_with_backup(command, root, run, destination, seconds, label="Colab local")
def snapshot_run(run, destination, force=False):
    return _snapshot_run(run, destination, force, label="Colab local")


## Reprendre un ZIP déjà téléchargé (optionnel)

Laisser RESTORE_RESULT=False pour la première passe.


In [ ]:
if RESTORE_RESULT:
    from google.colab import files
    print("Choisir un CyDSP-HRTF-L4-Results.zip déjà téléchargé pour cette expérience.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choisir un seul ZIP de résultats.")
    value = next(iter(uploaded.values()))
    if len(value) > 64 * 1024 * 1024:
        raise ValueError("Résultats trop volumineux.")
    archive = RESULTS_ROOT / "imported-results.zip"
    atomic_bytes(archive, value)
    restore_direct_result(archive, RESULTS_ROOT / "experiments" / EXPERIMENT)
    print("Checkpoint vérifié et restauré. Les cellules suivantes reprennent le calcul.")
else:
    print("Nouvelle passe ou reprise des checkpoints déjà présents dans cette session.")


## 3. Installer l'environnement et vérifier le GPU
Python isolé : PyTorch **2.9.1 / CUDA 12.8**, NumPy **2.3.5**, h5py **3.15.1**.
Les bibliothèques CUDA sont téléchargées depuis l'index officiel PyTorch.
Le pilote est fourni par Colab : aucune installation de pilote système.
Installation initiale : plusieurs Go, sans copie dans Drive.

La L4 et les A100 utilisent la même recette float32. TF32 et AMP restent
désactivés pour conserver la précision de référence. Le GPU réel, sa mémoire
et le pilote sont affichés. REQUIRE_A100=False accepte la L4 et les autres GPU CUDA.

Ce profil exige une L4 par défaut. Le message GPU confirme le matériel alloué.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
VENV = ROOT / "build/colab-venv"
PYTHON = VENV / "bin/python"
if not PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV)], check=True)
# pip officiel --python gère aussi un environnement créé sans pip.
pip_command = [sys.executable, "-m", "pip", "--python", str(PYTHON)]
subprocess.run([*pip_command, "install", "--index-url", "https://pypi.org/simple",
                "pip==25.3", "-r", str(ROOT / "scripts/requirements-hrtf-training.txt")], check=True)
subprocess.run([*pip_command, "install", "torch==2.9.1",
                "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
probe = r"""
import json, torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA inaccessible : vérifier le runtime GPU Colab et son pilote")
p = torch.cuda.get_device_properties(0)
x = torch.ones(8, device='cuda')
assert float((x*x).sum()) == 8
torch.cuda.synchronize()
print(json.dumps({'gpu': p.name, 'memoryGiB': p.total_memory / 2**30,
                  'torch': str(torch.__version__), 'cuda': torch.version.cuda,
                  'capability': list(torch.cuda.get_device_capability(0))}))
"""
GPU = json.loads(subprocess.check_output([str(PYTHON), "-c", probe], text=True))
print(json.dumps(GPU, indent=2))
if REQUIRE_A100 and "A100" not in GPU["gpu"].upper():
    raise RuntimeError("Session sans A100. Choisir une A100 ou autoriser un autre GPU.")
subprocess.run([str(PYTHON), str(ROOT / "scripts/train_hrtf_pilot.py"), "--self-test"], cwd=ROOT, check=True)

if REQUIRE_L4 and "L4" not in GPU["gpu"].upper():
    raise RuntimeError("Une NVIDIA L4 est requise pour ce profil ; sélectionner la L4 dans Colab.")


## 4. Télécharger les mesures

Les mesures SADIE II D2 48 kHz originales sont téléchargées depuis Zenodo dans la session Colab. Tailles, hashes et CRC vérifiés. Aucun accès Google Drive.


In [ ]:
selected_sources = [s for s in catalog["subjects"]
                    if DOWNLOAD_ALL_SUBJECTS or s["subject"] in SUBJECTS]
CORPUS = ROOT / "build/hrtf-training/sadie-ii-v2-2"
archive_cache = RESULTS_ROOT / "cache/sadie-ii-v2-2"
for item in selected_sources:
    name = item["subject"] + "_HRIR_SOFA.zip"
    local, cached = CORPUS / "archives" / name, archive_cache / name
    if CACHE_ARCHIVES_LOCAL and not local.exists() and cached.is_file():
        atomic_bytes(local, cached.read_bytes())
command = [str(PYTHON), str(ROOT / "scripts/fetch_hrtf_training.py"), "--rate", str(SAMPLE_RATE)]
if not DOWNLOAD_ALL_SUBJECTS:
    command += ["--subjects", *SUBJECTS]
subprocess.run(command, cwd=ROOT, check=True)
if CACHE_ARCHIVES_LOCAL:
    for item in selected_sources:
        name = item["subject"] + "_HRIR_SOFA.zip"
        local, cached = CORPUS / "archives" / name, archive_cache / name
        value = local.read_bytes()
        if len(value) != item["archiveBytes"] or digest(value) != item["archiveSha256"]:
            raise ValueError("Archive locale invalide")
        if not cached.exists() or digest(cached.read_bytes()) != item["archiveSha256"]:
            atomic_bytes(cached, value)
            if digest(cached.read_bytes()) != item["archiveSha256"]:
                raise OSError("Échec de vérification du cache Drive")
print("Corpus local prêt pour", SUBJECTS)


## 5. Mesurer les batches sur ce GPU
Essai court sur les seules directions d'entraînement : **millisecondes par
mise à jour**, **exemples par seconde** et mémoire GPU. Il ne modifie pas
automatiquement BATCH_SIZE : un autre batch modifie aussi la convergence.
La projection pour 100 000 étapes est du calcul seul, hors installation,
données, validation et sauvegardes.

Le petit modèle peut rester limité par les lancements de kernels. Ces mesures
permettent d'évaluer le gain réel sans le déduire de la quantité de VRAM.


In [ ]:
if RUN_BATCH_BENCHMARK:
    benchmark_path = ROOT / "build/colab/batch-benchmark.json"
    subprocess.run([str(PYTHON), str(ROOT / "scripts/benchmark_hrtf_training.py"),
                    "--subject", SUBJECTS[0], "--sample-rate", str(SAMPLE_RATE), "--filters", str(max(FILTER_BUDGETS)),
                    "--iterations", "80", "--output", str(benchmark_path)], cwd=ROOT, check=True)
    benchmark = read_json(benchmark_path)
    atomic_json(RESULTS_ROOT / "benchmarks" / (EXPERIMENT + ".json"), benchmark)
    for row in benchmark["rows"]:
        if "millisecondsPerStep" in row:
            print(f"Batch {row['batch']}: {row['millisecondsPerStep']:.2f} ms/étape ; "
                  f"{row['samplesPerSecond']:,.0f} exemples/s ; "
                  f"{row['peakAllocatedGpuBytes']/2**30:.2f} Go GPU ; "
                  f"100k étapes ≈ {row['computeSecondsFor100000Steps']/60:.1f} min de calcul")
else:
    print("Mesure de batches ignorée.")


## 6. Affiner / reprendre

Les meilleurs poids initiaux 48 kHz restent éligibles. Même réseau, 140000 étapes, batch 512 et taux réduit. Sélection sur la validation. Poids, Adam et états aléatoires sont enregistrés localement dans Colab avec des snapshots périodiques. Après une interruption sur la même machine, relancer cette cellule. Télécharger un ZIP depuis la dernière cellule avant de libérer la machine.


In [ ]:
PERSISTENT = RESULTS_ROOT / "experiments" / EXPERIMENT
plan = {"schema": 1, "sourceBundleSha256": EXPECTED_BUNDLE_SHA256, "subjects": SUBJECTS,
        "filterBudgets": FILTER_BUDGETS, "steps": STEPS, "batchSize": BATCH_SIZE,
        "validationEvery": VALIDATION_EVERY, "warmStartD2": WARM_START_D2,
        "width": 96, "featureBands": 4, "centerGrid": "hybrid",
        "frequencyGrid": "mixed", "selectionGrid": "linear",
        "storage": "colab-local-download", "precision": "float32; TF32 off; AMP off", "sampleRate": SAMPLE_RATE,
        "learningRate": LEARNING_RATE, "minimumLearningRate": MINIMUM_LEARNING_RATE}
plan_path = PERSISTENT / "plan.json"
if plan_path.exists() and read_json(plan_path) != plan:
    raise ValueError("Le plan de cette expérience diffère. Garder le plan de reprise ou choisir un autre EXPERIMENT.")
atomic_json(plan_path, plan)
RUNS = []
for subject in SUBJECTS:
    for filters in FILTER_BUDGETS:
        name = f"{EXPERIMENT}-{subject}-f{filters}"
        run = ROOT / "build/hrtf-runs" / name
        backup = PERSISTENT / "runs" / name
        if not (run / "resume.pt").is_file():
            restore_run(backup, run)
        info = read_json(run / "resume-info.json") if (run / "resume-info.json").is_file() else {}
        done = ((run / "report.json").is_file()
                and read_json(run / "report.json").get("status") == "completed"
                and info.get("step") == STEPS)
        if done:
            print(name, ": déjà terminé.")
        else:
            trainer = [str(PYTHON), str(ROOT / "scripts/train_hrtf_pilot.py")]
            if (run / "resume.pt").is_file():
                command = [*trainer, "--resume-run", str(run)]
            else:
                command = [*trainer, "--subject", subject, "--filters", str(filters), "--width", "96",
                           "--feature-bands", "4", "--center-grid", "hybrid",
                           "--frequency-grid", "mixed", "--selection-grid", "linear",
                           "--sample-rate", str(SAMPLE_RATE),
                           "--learning-rate", str(LEARNING_RATE),
                           "--minimum-learning-rate", str(MINIMUM_LEARNING_RATE),
                           "--steps", str(STEPS), "--batch-size", str(BATCH_SIZE),
                           "--validation-every", str(VALIDATION_EVERY),
                           "--validation-only", "--output-dir", str(run)]
                reference = REFERENCES.get(str(filters))
                if WARM_START_D2 and subject == "D2":
                    if reference is None:
                        raise ValueError("Poids D2 absents du paquet ; désactiver WARM_START_D2.")
                    command += ["--initial-weights", str(ROOT / reference["checkpoint"])]
            run_with_backup(command, ROOT, run, backup, SAVE_EVERY_SECONDS)
        RUNS.append({"subject": subject, "filters": filters, "run": run, "backup": backup})
print("Entraînements terminés :", PERSISTENT)

print("Rythme et résultats par modèle :")
for item in RUNS:
    result = read_json(item["run"] / "report.json")
    initial, best = result["initialValidation"], result["bestValidation"]
    print(item["subject"], "validation linéaire :", initial["validationLinearRmseDb"], "→", best["validationLinearRmseDb"], "dB RMS ; meilleur pas", result["bestStep"])


## 7. Figer les modèles puis évaluer
Les hashes de tous les meilleurs poids sont figés avant l'évaluation finale.
Rapports : erreurs spectrales en dB sur grilles log et linéaire, erreur ITD en
microsecondes, voisin mesuré, pôles stables, oracle RBJ complexe float64,
accord CPU/GPU et compensation du retard propre aux filtres.
Une erreur plus basse est préférable.

Découpage 80/10/10 des directions d'un auditeur connu. D2 avec poids de référence
déjà évalués : suivi du même test, pas un nouveau test indépendant.
Ne pas sélectionner des réglages sur ces mêmes chiffres de test. Le renderer
mesuré reste la référence ; entraîner sur GPU ne démontre pas un gain de vitesse
de rendu audio CPU.


In [ ]:
selection = {"plan": plan, "checkpoints": [
    {"subject": item["subject"], "filters": item["filters"],
     "run": item["run"].relative_to(ROOT).as_posix(),
     "sha256": digest((item["run"] / "best.pt").read_bytes()),
     "testPreviouslyUsedByReference": WARM_START_D2 and item["subject"] == "D2"}
    for item in RUNS]}
frozen_path = PERSISTENT / "frozen-selection.json"
if frozen_path.exists() and read_json(frozen_path) != selection:
    raise ValueError("Sélection figée modifiée ; conserver les résultats et créer une autre expérience.")
atomic_json(frozen_path, selection)
for item, checkpoint in zip(RUNS, selection["checkpoints"]):
    run, backup = item["run"], item["backup"]
    if digest((run / "best.pt").read_bytes()) != checkpoint["sha256"]:
        raise ValueError("Poids modifiés après gel")
    atomic_json(run / "frozen-selection.json", selection)
    if RUN_FINAL_EVALUATION:
        for script, flag in (("train_hrtf_pilot.py", "--evaluate-run"), ("check_hrtf_field.py", "--run")):
            run_with_backup([str(PYTHON), str(ROOT / "scripts" / script), flag, str(run)],
                            ROOT, run, backup, SAVE_EVERY_SECONDS)
        result = read_json(run / "evaluation.json")
        print(item["subject"], item["filters"], "filtres :",
              f"log {result['heldOutTestLogFrequency']['magnitudeRmseDb']:.3f} dB ;",
              f"linéaire {result['heldOutTestLinearFrequency']['magnitudeRmseDb']:.3f} dB ;",
              f"ITD {result['heldOutTestLinearFrequency']['itdRmseUs']:.3f} µs")
    snapshot_run(run, backup, force=True)
print("Résultats sauvegardés :", PERSISTENT)


## 8. Lire les courbes

Les métriques sont en dB RMS et µs RMS ; plus bas est préférable. La cellule suivante télécharge les résultats complets. Les checkpoints restent locaux tant que le ZIP n’a pas été téléchargé.


In [ ]:
import csv
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for item in RUNS:
    with (item["run"] / "progress.csv").open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    steps = [int(row["step"]) for row in rows]
    label = f"{item['subject']} / {item['filters']} filtres"
    axes[0].plot(steps, [float(row["validationLinearRmseDb"]) for row in rows], label=label)
    axes[1].plot(steps, [float(row["itdRmseUs"]) for row in rows], label=label)
for ax in axes:
    ax.set_xlabel("Mises à jour")
    ax.grid(alpha=.25)
    ax.legend()
axes[0].set_ylabel("Erreur spectrale de validation, grille linéaire (dB RMS)")
axes[1].set_ylabel("Erreur ITD de validation (µs RMS)")
fig.tight_layout()
plt.show()
print("Dossier des résultats Colab :", PERSISTENT)
for item in RUNS:
    pointer = read_json(item["backup"] / "latest.json")
    print(item["subject"], item["filters"], "filtres →", item["backup"] / pointer["archive"])


## 9. Télécharger les résultats

Cette cellule se lance à la fin. Elle peut aussi être exécutée après une interruption pour récupérer le dernier checkpoint complet. Le ZIP est à conserver sur ton PC ; il pourra servir à la reprise et à la comparaison du nouveau modèle.


In [ ]:
from google.colab import files

persistent = RESULTS_ROOT / "experiments" / EXPERIMENT
# Refresh local complete checkpoints; safe to run after a training interruption.
for pointer in sorted((persistent / "runs").glob("*/latest.json")):
    run = ROOT / "build/hrtf-runs" / pointer.parent.name
    snapshot_run(run, pointer.parent, force=True)
output = RESULTS_ROOT / "CyDSP-HRTF-L4-Results.zip"
export_direct_result(persistent, output)
print("Résultats prêts :", output, "—", output.stat().st_size, "octets")
print("Conserver ce fichier sur ton PC avant de fermer/supprimer la session Colab.")
files.download(str(output))
